# Human time-travel matrix (release×release) — cache build

This notebook builds a **cache-first** release×release matrix that is reused by downstream analysis notebooks to generate manuscript-ready figures.

## What this tests (marketing angle)

IDTrack’s core differentiator is that identifier conversion is explicitly parameterized by **Ensembl time** (release), rather than implicitly depending on “latest”.

This experiment measures, across many `from_release × to_release` combinations, how often Ensembl gene identifiers map as:

- **1→0**: no conversion possible under the chosen snapshot boundary
- **1→1**: a single target (stable mapping)
- **1→n**: multiple plausible targets (legitimate ambiguity from history)

It also measures a **drift signal**: how often the identifier *changes* across time travel even when mapping is 1→1.

## Manageability / reproducibility design

- Uses a **fixed release grid** (editable)
- Samples ENSG IDs **active at each `from_release`** via a single-pass multi-reservoir sampler
- Uses lightweight **bootstraps** (small default) so the grid is runnable on Slurm without exploding runtime
- Writes a **fingerprinted cache** (parameter hash embedded in filenames)

## Outputs

- Cache directory: `idtrack/docs/_notebooks/idtrack_cache/experiments/time_travel_matrix/`
- Figures/tables are produced by: `idtrack/reproducibility/experiments/experiment_time_travel_matrix/01_analyze_time_travel_matrix.ipynb`

## Notes

- This notebook can be memory-intensive because it loads the IDTrack graph. Recommended to run on Slurm.
- This is **not** an accuracy benchmark; it is a stability/ambiguity audit across the time axis.


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    notebook_context,
    stable_hash,
    write_json,
    write_pickle,
)

from idtrack_results import summarize_binned_conversion  # noqa: E402

ctx = notebook_context("time_travel_matrix", start=REPO_ROOT)

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache

print("IDTRACK_LOCAL_REPO:", IDTRACK_LOCAL_REPO)
print("CACHE_DIR:", CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISM_ALIAS = "human"

# Snapshot boundary: must be <= what you have cached locally.
SNAPSHOT_RELEASE = 114

# Release grid for the square matrix (edit freely; keep it manageable).
# Tip: a coarse grid (step=5) is usually enough for a clear Results figure.
RELEASES = list(range(75, 115, 5))
FROM_RELEASES = RELEASES
TO_RELEASES = RELEASES

# Sampling & bootstraps
N_POOL_PER_FROM = 2000   # reservoir pool size per from_release
N_IDS_PER_PAIR = 400     # evaluated IDs per (from_release, to_release, bootstrap)
N_BOOTSTRAPS = 4
RANDOM_SEED = 0

# Mapping semantics
STRATEGY = "all"  # expose ambiguity (1→n)

# Final targets (human-focused): Ensembl backbone + HGNC + UniProt.
# Note: HGNC is human-specific; UniProt is included for external matching.
FINAL_DATABASES = [
    None,
    "HGNC Symbol",
    "UniProtKB/Swiss-Prot",
]

PARAMS = {
    "organism_alias": ORGANISM_ALIAS,
    "snapshot_release": int(SNAPSHOT_RELEASE),
    "from_releases": [int(x) for x in FROM_RELEASES],
    "to_releases": [int(x) for x in TO_RELEASES],
    "n_pool_per_from": int(N_POOL_PER_FROM),
    "n_ids_per_pair": int(N_IDS_PER_PAIR),
    "n_bootstraps": int(N_BOOTSTRAPS),
    "random_seed": int(RANDOM_SEED),
    "strategy": str(STRATEGY),
    "final_databases": [db if db is not None else None for db in FINAL_DATABASES],
}

FP = stable_hash(json.dumps(PARAMS, sort_keys=True), n=12)
RESULTS_PKL = CACHE_DIR / f"time_travel_matrix_grid_{FP}.pickle"
PARAMS_JSON = CACHE_DIR / f"time_travel_matrix_params_{FP}.json"
POOLS_JSON = CACHE_DIR / f"time_travel_matrix_pools_{FP}.json"

print("Fingerprint:", FP)
print("RESULTS_PKL:", RESULTS_PKL)
print("PARAMS_JSON:", PARAMS_JSON)
print("POOLS_JSON:", POOLS_JSON)


In [ ]:
# -------------------- Build cache (graph + pools + grid summaries) --------------------

if RESULTS_PKL.exists() and PARAMS_JSON.exists() and POOLS_JSON.exists():
    print("Using existing cache:", RESULTS_PKL)
    grid = pd.read_pickle(RESULTS_PKL)
    display(grid.head())
else:
    import idtrack
    from idtrack._the_graph import TheGraph

    rng = np.random.default_rng(RANDOM_SEED)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = int(SNAPSHOT_RELEASE)
    if snapshot > int(latest):
        raise ValueError(f"snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}")

    if max(FROM_RELEASES + TO_RELEASES) > snapshot:
        raise ValueError(f"Release grid exceeds snapshot={snapshot}: max={max(FROM_RELEASES + TO_RELEASES)}")

    print(f"Building/loading graph: {organism} snapshot_release={snapshot}")
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)
    g = api.track.graph

    # Multi-reservoir pool sampling: one pass over ENSG nodes, keep a pool for each from_release.
    pools: dict[int, list[str]] = {int(r): [] for r in FROM_RELEASES}
    seen: dict[int, int] = {int(r): 0 for r in FROM_RELEASES}

    def _reservoir_update(r: int, item: str) -> None:
        seen[r] += 1
        if len(pools[r]) < int(N_POOL_PER_FROM):
            pools[r].append(item)
            return
        j = int(rng.integers(0, seen[r]))
        if j < int(N_POOL_PER_FROM):
            pools[r][j] = item

    t0_pool = time.perf_counter()
    for node in g.nodes:
        if not isinstance(node, str) or not node.startswith("ENSG"):
            continue
        try:
            ranges = g.get_active_ranges_of_id[node]
        except Exception:
            continue
        for r in FROM_RELEASES:
            rr = int(r)
            if TheGraph.is_point_in_range(ranges, rr):
                _reservoir_update(rr, node)

    dt_pool = time.perf_counter() - t0_pool
    print(f"Built pools in {dt_pool:.1f}s")

    pool_report = []
    for r in FROM_RELEASES:
        rr = int(r)
        pool_report.append({"from_release": rr, "seen_active": int(seen[rr]), "pool_size": int(len(pools[rr]))})
    pool_report_df = pd.DataFrame(pool_report).sort_values("from_release")
    display(pool_report_df)

    write_json(PARAMS, PARAMS_JSON)
    write_json({"pools": pools, "seen": seen, "pool_report": pool_report}, POOLS_JSON)

    # Grid compute
    rows: list[dict] = []
    for b in range(int(N_BOOTSTRAPS)):
        rng_b = np.random.default_rng(int(RANDOM_SEED) + 10_000 + b)
        for fr in FROM_RELEASES:
            fr = int(fr)
            pool = pools.get(fr, [])
            if not pool:
                continue
            ids = rng_b.choice(pool, size=min(int(N_IDS_PER_PAIR), len(pool)), replace=False).tolist()

            for to in TO_RELEASES:
                to = int(to)
                for final_db in FINAL_DATABASES:
                    final_label = final_db if final_db is not None else "Ensembl gene"

                    t0 = time.perf_counter()
                    matchings = api.convert_identifier_multiple(
                        ids.copy(),
                        from_release=fr,
                        to_release=to,
                        final_database=final_db,
                        strategy=STRATEGY,
                        verbose=False,
                    )
                    dt = time.perf_counter() - t0

                    bins = api.classify_multiple_conversion(matchings)
                    stats = summarize_binned_conversion(bins)

                    rows.append(
                        {
                            "bootstrap": int(b),
                            "from_release": int(fr),
                            "to_release": int(to),
                            "final_database": str(final_label),
                            "seconds": float(dt),
                            **stats,
                        }
                    )

    grid = pd.DataFrame(rows)
    write_pickle(grid, RESULTS_PKL)
    print("Wrote:", RESULTS_PKL)
    display(grid.head())


## Cheap integrity checks (recommended)

These checks do **not** load the graph again and do **not** recompute conversions.
They are meant to fail fast if something went wrong (e.g., corrupted cache, wrong parameterization).

**Sanity expectations:**

- The diagonal (`from_release == to_release`) should be dominated by 1→1 success on the Ensembl backbone.
- External targets (HGNC/UniProt) may introduce controlled ambiguity (1→n) and occasional ATM, but 1→0 should remain relatively low for modern releases.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

if 'grid' not in globals() or grid is None or grid.empty:
    raise RuntimeError('Grid is missing/empty. Did the cache build succeed?')

df = grid.copy()
den = df['total'].replace(0, np.nan)
df['frac_1_to_0'] = df['1_to_0'] / den
df['frac_tdm_total'] = (df['1_to_1_tdm'] + df['1_to_n_tdm']) / den
df['frac_atm_total'] = (df['1_to_1_atm'] + df['1_to_n_atm']) / den
df['sec_per_id'] = df['seconds'] / den

diag = df[df['from_release'].astype(int) == df['to_release'].astype(int)]

diag_summary = (
    diag.groupby('final_database', as_index=False)[
        ['frac_1_to_0', 'frac_tdm_total', 'frac_atm_total', 'sec_per_id']
    ]
    .mean(numeric_only=True)
    .sort_values('final_database')
)

print('Diagonal sanity summary (mean over releases):')
display(diag_summary)

out_diag = CACHE_DIR / f'time_travel_matrix_diagonal_sanity_{FP}.csv'
atomic_write_dataframe_csv(diag_summary, out_diag, index=False)
print('Wrote:', out_diag)
